In [ ]:
!pip install -q requests Pillow ImageHash

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
from PIL import Image, ImageOps
from io import BytesIO
from datetime import datetime, timezone
from google.colab import userdata
from getpass import getpass

import requests
import imagehash
import hashlib
import csv
import re
import time

THESIS_DIR = Path("/content/drive/MyDrive/College/Thesis")
REFERENCE_DIR = THESIS_DIR / "dataset"
RESULT_DIR = THESIS_DIR / "results_no_gofood"
MANIFEST = RESULT_DIR / "manifest.csv"

EXPECTED_REFERENCES = 21
RESULTS_PER_REFERENCE = 50
EXPECTED_TOTAL = EXPECTED_REFERENCES * RESULTS_PER_REFERENCE

PHASH_THRESHOLD = 4

MIN_WIDTH = 256
MIN_HEIGHT = 256
MAX_DOWNLOAD_SIZE = 20 * 1024 * 1024

SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

MAX_SEARCHES_PER_REFERENCE = 3

BLOCKED_DOMAINS = {
    "gofood.co.id",
    "gojek.com",
    "go-jek.com",
    "gojekapi.com",
}


def is_blocked_url(url):
    if not url:
        return False

    try:
        hostname = (
            urlparse(url).hostname or ""
        ).lower().rstrip(".")
    except (TypeError, ValueError):
        return False

    return any(
        hostname == domain
        or hostname.endswith("." + domain)
        for domain in BLOCKED_DOMAINS
    )


def is_blocked_match(match):
    return any(
        is_blocked_url(match.get(field))
        for field in ("image", "thumbnail", "link")
    )

RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Reference directory:", REFERENCE_DIR)
print("Result directory:", RESULT_DIR)
print("Expected total:", EXPECTED_TOTAL)

In [ ]:
try:
    SERPAPI_KEY = userdata.get("SERPAPI_KEY")
except Exception:
    SERPAPI_KEY = None

if not SERPAPI_KEY:
    SERPAPI_KEY = getpass("Enter your SERPAPI_KEY: ").strip()

if not SERPAPI_KEY:
    raise RuntimeError("SERPAPI_KEY is required.")

print("SerpApi key loaded.")

In [ ]:
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(X11; Linux x86_64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/127 Safari/537.36"
    )
})

print("HTTP session created.")

In [ ]:
def load_image(source):
    # Load from a path or raw bytes and normalize to RGB.
    if isinstance(source, (str, Path)):
        img = Image.open(source)
    else:
        img = Image.open(BytesIO(source))

    img.load()
    img = ImageOps.exif_transpose(img)

    if img.mode in ("RGBA", "LA"):
        rgba = img.convert("RGBA")
        background = Image.new("RGB", rgba.size, "white")
        background.paste(rgba, mask=rgba.getchannel("A"))
        return background

    return img.convert("RGB")


def exact_hash(img):
    # SHA256 based on decoded RGB pixels.
    h = hashlib.sha256()
    h.update(f"{img.width}x{img.height}:RGB".encode())
    h.update(img.tobytes())
    return h.hexdigest()


def perceptual_hash(img):
    # Detect resized/recompressed near-duplicates.
    return imagehash.phash(img)


def safe_filename(name):
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return name.strip("._-") or "reference"


print("Image utility functions loaded.")

In [ ]:
seen_exact_hashes = set()
seen_phashes = []
seen_urls = set()


def register_image(img):
    seen_exact_hashes.add(exact_hash(img))
    seen_phashes.append(perceptual_hash(img))


def check_duplicate(img):
    candidate_exact = exact_hash(img)

    if candidate_exact in seen_exact_hashes:
        return "exact duplicate"

    candidate_phash = perceptual_hash(img)

    for existing_phash in seen_phashes:
        distance = candidate_phash - existing_phash

        if distance <= PHASH_THRESHOLD:
            return f"near duplicate (pHash distance={distance})"

    return None


print("Duplicate detection system loaded.")

In [ ]:
def prepare_lens_image(path):
    # Create a temporary compressed copy; original Drive image stays untouched.
    img = load_image(path)

    img.thumbnail(
        (1800, 1800),
        Image.Resampling.LANCZOS
    )

    quality = 92

    while True:
        buffer = BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=quality,
            optimize=True
        )

        data = buffer.getvalue()

        if len(data) <= 490_000:
            return data

        if quality > 52:
            quality -= 8
        else:
            new_width = max(320, int(img.width * 0.85))
            new_height = max(320, int(img.height * 0.85))

            img = img.resize(
                (new_width, new_height),
                Image.Resampling.LANCZOS
            )

            quality = 82


print("Lens image preparation loaded.")

In [ ]:
def upload_to_lens(reference_path):
    image_bytes = prepare_lens_image(reference_path)

    response = session.post(
        "https://serpapi.com/image",
        data={"api_key": SERPAPI_KEY},
        files={
            "image": (
                "reference.jpg",
                image_bytes,
                "image/jpeg"
            )
        },
        timeout=60
    )

    response.raise_for_status()
    data = response.json()

    if data.get("error"):
        raise RuntimeError(data["error"])

    image_id = data.get("image_id")

    if not image_id:
        raise RuntimeError("SerpApi did not return image_id.")

    return image_id


def lens_search(image_id, auto_crop=False, query=None):
    params = {
        "engine": "google_lens",
        "image_id": image_id,
        "type": "visual_matches",
        "api_key": SERPAPI_KEY,
        "country": "id",
        "hl": "id",
        "safe": "active",
        "auto_crop": str(auto_crop).lower(),
    }

    if query:
        params["q"] = query

    response = session.get(
        "https://serpapi.com/search.json",
        params=params,
        timeout=90
    )

    response.raise_for_status()
    data = response.json()

    if data.get("error"):
        raise RuntimeError(data["error"])

    return data


def get_related_queries(data):
    queries = []

    for item in data.get("related_content", []):
        query = item.get("query")

        if query and query not in queries:
            queries.append(query)

    return queries


print("Google Lens functions loaded.")

In [ ]:
def download_bytes(url, referer=None):
    if is_blocked_url(url):
        raise ValueError(f"Blocked image domain: {url}")

    headers = {}

    if referer:
        headers["Referer"] = referer

    with session.get(
        url,
        headers=headers,
        stream=True,
        timeout=(12, 35),
        allow_redirects=True
    ) as response:
        response.raise_for_status()

        if is_blocked_url(response.url):
            raise ValueError(
                f"Redirected to blocked domain: {response.url}"
            )

        declared_size = response.headers.get("Content-Length")

        if declared_size and int(declared_size) > MAX_DOWNLOAD_SIZE:
            raise ValueError("Image too large.")

        chunks = []
        downloaded = 0

        for chunk in response.iter_content(chunk_size=128 * 1024):
            if not chunk:
                continue

            downloaded += len(chunk)

            if downloaded > MAX_DOWNLOAD_SIZE:
                raise ValueError("Image exceeded download size limit.")

            chunks.append(chunk)

    return b"".join(chunks)


def download_match(match):
    if is_blocked_match(match):
        raise ValueError("Match contains a blocked GoFood/Gojek domain.")

    possible_urls = []

    original = match.get("image")
    thumbnail = match.get("thumbnail")

    if original:
        possible_urls.append(original)

    if thumbnail:
        possible_urls.append(thumbnail)

    last_error = None

    for url in possible_urls:
        if is_blocked_url(url):
            last_error = ValueError(
                f"Blocked image domain: {url}"
            )
            continue

        if url in seen_urls:
            continue

        try:
            raw = download_bytes(
                url,
                referer=match.get("link")
            )

            img = load_image(raw)

            if img.width < MIN_WIDTH or img.height < MIN_HEIGHT:
                raise ValueError(
                    f"Image too small: {img.width}x{img.height}"
                )

            return img, url

        except Exception as error:
            last_error = error

    if last_error:
        raise last_error

    raise ValueError("No usable image URL.")


print("Download functions loaded with domain blacklist.")


In [ ]:
MANIFEST_COLUMNS = [
    "reference",
    "output",
    "image_url",
    "source_page",
    "source",
    "title",
    "saved_at",
]


def load_existing_urls():
    if not MANIFEST.exists():
        return

    try:
        with MANIFEST.open(
            "r",
            newline="",
            encoding="utf-8"
        ) as file:
            reader = csv.DictReader(file)

            for row in reader:
                url = row.get("image_url")

                if url:
                    seen_urls.add(url)

    except Exception as error:
        print("Could not read manifest:", error)


def save_manifest_row(row):
    already_exists = MANIFEST.exists()

    with MANIFEST.open(
        "a",
        newline="",
        encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MANIFEST_COLUMNS
        )

        if not already_exists:
            writer.writeheader()

        writer.writerow(row)


print("Manifest functions loaded.")

In [ ]:
reference_paths = sorted([
    path
    for path in REFERENCE_DIR.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
])

print("Found", len(reference_paths), "reference images.")

for index, path in enumerate(reference_paths, start=1):
    print(f"{index:02d}.", path.name)

if len(reference_paths) != EXPECTED_REFERENCES:
    raise RuntimeError(
        f"Expected {EXPECTED_REFERENCES} images in {REFERENCE_DIR}, "
        f"but found {len(reference_paths)}."
    )

In [ ]:
references = []
original_hashes = {}

for path in reference_paths:
    img = load_image(path)
    ehash = exact_hash(img)

    if ehash in original_hashes:
        raise RuntimeError(
            "Duplicate reference image detected:\n"
            f"{original_hashes[ehash]}\n"
            f"{path}"
        )

    original_hashes[ehash] = path

    prefix = (
        f"{safe_filename(path.stem)}_"
        f"{ehash[:8]}"
    )

    references.append({
        "path": path,
        "prefix": prefix,
    })

    register_image(img)

print("All reference images are unique.")
print(len(references), "references registered.")

In [ ]:
load_existing_urls()

existing_images = [
    path
    for path in RESULT_DIR.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
]

print("Existing result images:", len(existing_images))
print("Building duplicate index...")

for index, path in enumerate(existing_images, start=1):
    try:
        img = load_image(path)
        register_image(img)

    except Exception as error:
        print("Could not index:", path.name, error)

    if index % 100 == 0:
        print(f"Indexed {index}/{len(existing_images)}")

print("Existing result index ready.")

In [ ]:
def existing_for_reference(prefix):
    return len(
        list(
            RESULT_DIR.glob(
                f"{prefix}__*.jpg"
            )
        )
    )


def next_output_file(prefix, number):
    while True:
        output = RESULT_DIR / (
            f"{prefix}__{number:03d}.jpg"
        )

        if not output.exists():
            return output

        number += 1


print("Output helpers loaded.")

In [ ]:
for ref_number, reference in enumerate(
    references,
    start=1
):
    path = reference["path"]
    prefix = reference["prefix"]

    saved = existing_for_reference(prefix)

    print()
    print("=" * 70)
    print(f"REFERENCE {ref_number}/{len(references)}")
    print("File:", path.name)
    print(f"Downloaded: {saved}/{RESULTS_PER_REFERENCE}")

    if saved >= RESULTS_PER_REFERENCE:
        print("Already complete.")
        continue

    try:
        print("Uploading reference to Google Lens...")
        image_id = upload_to_lens(path)

    except Exception as error:
        print("Lens upload failed:", error)
        continue

    search_queue = [
        {
            "auto_crop": False,
            "query": None
        },
        {
            "auto_crop": True,
            "query": None
        },
    ]

    related_queries = []
    searched = 0
    already_attempted = set()

    while (
        saved < RESULTS_PER_REFERENCE
        and searched < MAX_SEARCHES_PER_REFERENCE
    ):
        if search_queue:
            options = search_queue.pop(0)

        elif related_queries:
            options = {
                "auto_crop": False,
                "query": related_queries.pop(0),
            }

        else:
            break

        try:
            print(f"\nRunning Lens search #{searched + 1}...")

            results = lens_search(
                image_id,
                **options
            )

            searched += 1

        except Exception as error:
            print("Lens search failed:", error)
            break

        matches = results.get(
            "visual_matches",
            []
        )

        print("Candidates found:", len(matches))

        for query in get_related_queries(results):
            if query not in related_queries:
                related_queries.append(query)

        for match in matches:
            if saved >= RESULTS_PER_REFERENCE:
                break

            if is_blocked_match(match):
                blocked_hosts = sorted({
                    urlparse(url).hostname
                    for url in (
                        match.get("image"),
                        match.get("thumbnail"),
                        match.get("link"),
                    )
                    if url and is_blocked_url(url)
                })

                print(
                    "BLOCKED DOMAIN:",
                    ", ".join(blocked_hosts)
                )
                continue

            candidate_url = (
                match.get("image")
                or match.get("thumbnail")
            )

            if not candidate_url:
                continue

            if candidate_url in already_attempted:
                continue

            already_attempted.add(candidate_url)

            if candidate_url in seen_urls:
                continue

            try:
                img, used_url = download_match(match)

                if is_blocked_url(used_url):
                    raise ValueError(
                        f"Blocked downloaded URL: {used_url}"
                    )

                duplicate = check_duplicate(img)

                if duplicate:
                    seen_urls.add(used_url)
                    print("DUPLICATE:", duplicate)
                    continue

                output_path = next_output_file(
                    prefix,
                    saved + 1
                )

                img.save(
                    output_path,
                    "JPEG",
                    quality=95,
                    optimize=True
                )

                register_image(img)
                seen_urls.add(used_url)

                saved += 1

                save_manifest_row({
                    "reference": path.name,
                    "output": output_path.name,
                    "image_url": used_url,
                    "source_page": match.get("link", ""),
                    "source": match.get("source", ""),
                    "title": match.get("title", ""),
                    "saved_at": datetime.now(
                        timezone.utc
                    ).isoformat(),
                })

                print(
                    f"SAVED "
                    f"{saved:02d}/{RESULTS_PER_REFERENCE}"
                    f" -> {output_path.name}"
                )

            except Exception as error:
                print(
                    "SKIPPED:",
                    type(error).__name__,
                    error
                )

            time.sleep(0.1)

    print()

    if saved >= RESULTS_PER_REFERENCE:
        print(
            f"COMPLETE: "
            f"{saved}/{RESULTS_PER_REFERENCE}"
        )
    else:
        print(
            f"WARNING: only "
            f"{saved}/{RESULTS_PER_REFERENCE} "
            "unique usable images were found."
        )


In [ ]:
final_images = [
    path
    for path in RESULT_DIR.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
]

print()
print("=" * 70)
print("CRAWL FINISHED")
print("=" * 70)

print("Reference images :", len(reference_paths))
print("Target/reference :", RESULTS_PER_REFERENCE)
print("Expected total   :", EXPECTED_TOTAL)
print("Actual results   :", len(final_images))

print()
print("Result directory:")
print(RESULT_DIR)

print()
print("Manifest:")
print(MANIFEST)

if len(final_images) == EXPECTED_TOTAL:
    print()
    print("SUCCESS:")
    print("32 references × 32 results = 1024 images.")

elif len(final_images) < EXPECTED_TOTAL:
    print()
    print("The target has not been fully reached.")
    print(
        "Some reference images may not have produced "
        "32 unique downloadable visual matches."
    )

else:
    print()
    print("There are more than 1024 images in results.")
    print(
        "This probably means results from an older run "
        "are also present."
    )

## Cell 18 — Optional progress report

In [ ]:
print(f"{'Reference':45} {'Count':>5}")
print("-" * 53)

total = 0

for reference in references:
    count = existing_for_reference(
        reference["prefix"]
    )

    total += count

    print(
        f"{reference['path'].name[:45]:45} "
        f"{count:>5}"
    )

print("-" * 53)
print(f"{'TOTAL':45} {total:>5}")

print()
print(f"Target: {EXPECTED_TOTAL}")

In [ ]:
import shutil

source_folder = "/content/drive/MyDrive/College/Thesis/results"
output_zip = "/content/thesis_results"

shutil.make_archive(
    output_zip,
    "zip",
    source_folder
)

print("Created:", output_zip + ".zip")

In [ ]:
from google.colab import files

files.download("/content/thesis_results.zip")